In [ ]:
import source.schedulers as schedulers
import source.neural_network as NN
import source.activation_functions as activation_functions
import source.cost_functions as cost_functions
import source.utils as utils
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import source.plotting as plotting

In [ ]:
# Custom imports
from source.mnist_preprocessing import ITERATIONS, MNIST_RANDOM_STATE, TORCH_SEED
from source.mnist_preprocessing import  ETA_VALUES
from source.mnist_preprocessing import MOMENTUM,NP_RANDOM_SEED

In [ ]:
MNIST_RANDOM_STATE = 42
TEST_SPLIT = 0.2
# Download MNIST dataset
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

# Extract data (features) and target (labels)
X = mnist.data
y = mnist.target

# Scaling pixel values
X = X / 255.0

enc = OneHotEncoder(sparse_output=False)
y_onehot = enc.fit_transform(y.reshape(-1,1))

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=TEST_SPLIT, random_state=MNIST_RANDOM_STATE)

In [ ]:
ACTIVATION_FUNCTION_OUTPUT = activation_functions.softmax
ACTIVATION_FUNCTION_OUTPUT_DERIVATIVE = None
cost_fn = cost_functions.class_multiclass_cross_entropy(l1=0, l2=0)

In [ ]:

EPOCHS = 100

BATCH_SIZE = 256
number_batch = int(np.floor(X_test.shape[0]/BATCH_SIZE))


cost_f = 'CrossEntropy'
weight_decay_val = 0 # no lambda

rho1_val = 0.9
rho2_val = 0.999

LAMBDA_VALUES = np.logspace(-2, -4, 4)

VERBOSE = True
IMG_DATA = True # ensure flatten in model generation for image classification
SAVE_FIGURE = True
SHOW_PLOT = True

In [ ]:
ACTIVATION_FUNCTION = activation_functions.RELU
ACTIVATION_FUNCTION_DERIVATIVE = activation_functions.RELU_derivative
HIDDEN_LAYERS = (8,)
pen = 'L1'

In [ ]:
activations, activations_derivative, _dim = utils.create_activations_layderdim(ACTIVATION_FUNCTION, ACTIVATION_FUNCTION_DERIVATIVE, 
                                                                        ACTIVATION_FUNCTION_OUTPUT,ACTIVATION_FUNCTION_OUTPUT_DERIVATIVE,
                                                                        HIDDEN_LAYERS, y_train, X_train)

In [ ]:

# Define model as input to loop to ensure new model each iteration
def model_classify_fn():
    return NN.NN(_dim, activations, activations_derivative, cost_fn, NP_RANDOM_SEED)


classify_results = utils.neural_network_loop(model_classify_fn,
                                              ETA_VALUES,
                                              LAMBDA_VALUES,
                                              'ADAM', 
                                              EPOCHS, 
                                              X_train, y_train, 
                                              X_test, y_test,
                                              batch_val=number_batch,
                                              rho=rho1_val,
                                              rho2=rho2_val,
                                              l1_l2=pen,
                                              verbose=True)

In [ ]:
def accuracy(prediction,target):
    assert prediction.size == target.size
    return np.average((target == prediction))

accuracy_list = []

for prediction in classify_results.Predictions:
    accuracy_list.append(accuracy(prediction,y_test))

applied_df = classify_results.assign(Accuracy=accuracy_list)

In [ ]:

task = 'part_f'
hidden_str = 'hidden_8'
algo = 'ownNN_RELU_L1'
plot_type = 'heatmap'

filename_values = f'{task}-{plot_type}-{hidden_str}-iter{EPOCHS}-batch{number_batch}-{algo}'

plotting.plot_heatmap(applied_df, 'Accuracy',
              y_axis_scientific=True, 
              title='Heatmap with Accuracy values', 
             filename = 'classify_own' + filename_values)